# GenSivana Level 2 — v2 Clean OOF Hybrid Forensics

Classification reproductible des images en trois classes :

- `0` : photographie réelle ;
- `1` : image entièrement générée ;
- `2` : photographie localement retouchée.

La recette transpose au multiclasse la validation rigoureuse de la v7 du
Level 1. Trois experts complémentaires sont entraînés sur des folds groupés
par pHash : DeepGuard, ConvNeXt-V2 RGB et ConvNeXt-V2 NPR. Un stacker
multinomial et des biais de décision robustes sont appris uniquement à
partir des prédictions OOF. La validation officielle reste intacte jusqu'à
l'audit final.

Environnement cible : Kaggle, deux GPU T4, durée visée inférieure à sept
heures. Les caches permettent de reprendre un run interrompu.


In [1]:
%pip install -q imagehash timm transformers


Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import json
import math
import platform
import random
import time
import warnings
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
import sklearn
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from IPython.display import display
from PIL import Image
from PIL import ImageFile
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import AutoImageProcessor
from transformers import AutoModelForImageClassification
from transformers import get_cosine_schedule_with_warmup

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")

SEED = 20260726
N_CLASSES = 3
CLASS_NAMES = ["real", "generated", "local_edit"]
N_FOLDS = 3
META_FOLDS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()
AMP = DEVICE == "cuda"
OUT = Path("/kaggle/working")
if not OUT.exists():
    OUT = Path("./working_level2")
OUT.mkdir(exist_ok=True)

DEEP_NAME = "king1oo1/deepfake-model"
CNN_NAME = "convnextv2_tiny.fcmae_ft_in22k_in1k_384"
IMG_DEEP = 224
IMG_CNN = 384
# fix (kernel meurt / hang avec c10::AcceleratorError après le 1er expert,
# exactement quand un AUTRE modèle sollicite le 2e GPU pour la 1re fois) :
# même cause racine identifiée et corrigée sur le Level 1 (v7) -- le lien
# inter-GPU (P2P) entre les deux T4 de cet environnement Kaggle est instable
# avec nn.DataParallel. On désactive le multi-GPU (un seul flag ici, déjà
# bien centralisé dans le code). BATCH_*/ACCUM_* retombent automatiquement
# sur leurs valeurs "1 GPU" (12*4=48, 4*6=24), qui donnent EXACTEMENT le même
# batch effectif que les valeurs "2 GPU" (24*2=48, 8*3=24) -- donc la recette
# d'entraînement (poids, LR schedule, dynamique de gradient) ne change pas.
USE_TWO_GPUS = False
BATCH_DEEP = 24 if N_GPU >= 2 and USE_TWO_GPUS else 12
BATCH_CNN = 8 if N_GPU >= 2 and USE_TWO_GPUS else 4
ACCUM_DEEP = 2 if N_GPU >= 2 and USE_TWO_GPUS else 4
ACCUM_CNN = 3 if N_GPU >= 2 and USE_TWO_GPUS else 6
NUM_WORKERS = 2
EPOCHS_DEEP = 4
EPOCHS_CNN = 4
PATIENCE = 2
LABEL_SMOOTHING = 0.02
RUN_TAG = "level2_v2c_s20260726_f3"
FORCE_RETRAIN = False
START_TIME = time.monotonic()


def seed_all(seed=SEED):
    '''Seed Python, NumPy, PyTorch, CUDA, and DataLoader workers.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id):
    '''Give each DataLoader worker a deterministic independent seed.'''
    worker_seed = (SEED + worker_id) % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


seed_all()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("device", DEVICE, "GPUs", N_GPU, "output", OUT)


device cuda GPUs 2 output /kaggle/working


## 1. Données et groupes anti-doublons

Le train sert à construire les folds OOF. Le split `val` officiel est un
holdout : aucun modèle, biais ou hyperparamètre n'est choisi à partir de son
score. Les quasi-doublons du train sont réunis par pHash avant la séparation.


In [3]:
def find_root():
    '''Find the first directory containing the competition dataset.'''
    candidates = [
        Path(
            "/kaggle/input/competitions/"
            "gensivana-real-or-fake-level-2-classification"
        ),
        Path("/kaggle/input/gensivana-real-or-fake-level-2-classification"),
        Path("/content/gensivana-real-or-fake-level-2-classification"),
        Path("."),
    ]
    for root in candidates:
        required = [
            root / "train" / "ground_truth.csv",
            root / "val" / "ground_truth.csv",
            root / "test" / "images",
            root / "sample_submission.csv",
        ]
        if all(path.exists() for path in required):
            return root
    raise FileNotFoundError("Dataset Level 2 introuvable")


def read_split(root, name, labelled=True):
    '''Read one split, normalize labels, and validate image paths.'''
    image_dir = root / name / "images"
    if labelled:
        frame = pd.read_csv(root / name / "ground_truth.csv")
        if "classe" in frame and "class" not in frame:
            frame = frame.rename(columns={"classe": "class"})
        frame = frame.rename(columns={"class": "target"})
    else:
        sample = pd.read_csv(root / "sample_submission.csv")
        frame = sample[["image_id"]].copy()
    frame["path"] = frame.image_id.map(lambda x: image_dir / x)
    missing = [path for path in frame.path if not path.exists()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images manquantes")
    if labelled:
        labels = set(frame.target.astype(int).unique())
        if labels != {0, 1, 2}:
            raise ValueError(f"Labels inattendus : {labels}")
        frame.target = frame.target.astype(int)
    return frame.reset_index(drop=True)


ROOT = find_root()
train_df = read_split(ROOT, "train")
val_df = read_split(ROOT, "val")
test_df = read_split(ROOT, "test", labelled=False)
print(ROOT, len(train_df), len(val_df), len(test_df))
print(train_df.target.value_counts().sort_index().to_dict())
print(val_df.target.value_counts().sort_index().to_dict())

hash_cache = OUT / f"{RUN_TAG}_train_phash.npy"
if hash_cache.exists() and not FORCE_RETRAIN:
    hashes = np.load(hash_cache, allow_pickle=True).tolist()
else:
    hashes = []
    for path in train_df.path:
        with Image.open(path) as image:
            hashes.append(str(imagehash.phash(image.convert("RGB"))))
    np.save(hash_cache, np.array(hashes, dtype=object))

parent = np.arange(len(train_df))


def find(index):
    '''Return the canonical union-find representative of an index.'''
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index


def union(left, right):
    '''Merge two union-find components.'''
    root_left = find(left)
    root_right = find(right)
    if root_left != root_right:
        parent[root_right] = root_left


buckets = {}
for index, value in enumerate(hashes):
    integer = int(value, 16)
    keys = [
        (integer >> 51) & 8191,
        (integer >> 38) & 8191,
        (integer >> 25) & 8191,
        (integer >> 12) & 8191,
        integer & 4095,
    ]
    candidates = set()
    for position, key in enumerate(keys):
        candidates.update(buckets.get((position, key), []))
    for other in candidates:
        distance = (integer ^ int(hashes[other], 16)).bit_count()
        if distance <= 4:
            union(index, other)
    for position, key in enumerate(keys):
        buckets.setdefault((position, key), []).append(index)

groups = np.array([find(index) for index in range(len(train_df))])
cv_model = StratifiedGroupKFold(
    N_FOLDS,
    shuffle=True,
    random_state=SEED,
)
cv = list(cv_model.split(train_df, train_df.target, groups))
for fold, (train_index, valid_index) in enumerate(cv):
    overlap = set(groups[train_index]) & set(groups[valid_index])
    if overlap:
        raise RuntimeError(f"Fuite de groupes dans le fold {fold}")
    print(
        "fold",
        fold,
        "train",
        len(train_index),
        "valid",
        len(valid_index),
    )


/kaggle/input/competitions/gensivana-real-or-fake-level-2-classification 7857 978 993
{0: 2619, 1: 2619, 2: 2619}
{0: 326, 1: 326, 2: 326}
fold 0 train 5218 valid 2639
fold 1 train 5209 valid 2648
fold 2 train 5287 valid 2570


## 2. Transformations et représentations

DeepGuard travaille à 224 pixels. Les branches ConvNeXt-V2 utilisent 384
pixels. NPR retire une version moyennée et réagrandie de l'image pour mettre
en évidence les relations locales créées par les opérations de génération.


In [4]:
IM_MEAN = (0.485, 0.456, 0.406)
IM_STD = (0.229, 0.224, 0.225)
deep_processor = AutoImageProcessor.from_pretrained(DEEP_NAME)
DEEP_MEAN = tuple(deep_processor.image_mean)
DEEP_STD = tuple(deep_processor.image_std)


def make_transform(size, mean, std, train=False):
    '''Build deterministic evaluation or augmented training transforms.'''
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(
                size,
                scale=(0.55, 1.0),
                ratio=(0.75, 1.333),
                interpolation=transforms.InterpolationMode.BICUBIC,
            ),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([
                transforms.ColorJitter(0.1, 0.1, 0.06, 0.015)
            ], p=0.3),
            transforms.RandomChoice([
                transforms.Lambda(lambda image: image),
                transforms.GaussianBlur(3, (0.1, 0.8)),
            ]),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize(
            (size, size),
            interpolation=transforms.InterpolationMode.BICUBIC,
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


deep_train_tf = make_transform(
    IMG_DEEP,
    DEEP_MEAN,
    DEEP_STD,
    train=True,
)
deep_eval_tf = make_transform(IMG_DEEP, DEEP_MEAN, DEEP_STD)
cnn_train_tf = make_transform(
    IMG_CNN,
    IM_MEAN,
    IM_STD,
    train=True,
)
cnn_eval_tf = make_transform(IMG_CNN, IM_MEAN, IM_STD)


class ImageDataset(Dataset):
    '''Load RGB images and optional global plus local NPR views.'''

    def __init__(self, frame, transform, mode="rgb", multiview=False):
        '''Store the image table and input representation.'''
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.mode = mode
        self.multiview = multiview

    def __len__(self):
        '''Return the number of images in the table.'''
        return len(self.frame)

    def _represent(self, image):
        '''Transform one PIL image into an RGB or NPR tensor.'''
        tensor = self.transform(image)
        if self.mode != "npr":
            return tensor
        pooled = F.avg_pool2d(tensor[None], 2)
        upsampled = F.interpolate(
            pooled,
            size=tensor.shape[-2:],
            mode="nearest",
        )[0]
        return torch.tanh(2.5 * (tensor - upsampled))

    def __getitem__(self, index):
        '''Load one image, optional crops, target, and row index.'''
        with Image.open(self.frame.path.iloc[index]) as source:
            image = source.convert("RGB")
        views = [image]
        if self.multiview:
            width, height = image.size
            crop_width = max(32, int(0.72 * width))
            crop_height = max(32, int(0.72 * height))
            views.extend([
                image.crop((0, 0, crop_width, crop_height)),
                image.crop((
                    width - crop_width,
                    0,
                    width,
                    crop_height,
                )),
                image.crop((
                    0,
                    height - crop_height,
                    crop_width,
                    height,
                )),
                image.crop((
                    width - crop_width,
                    height - crop_height,
                    width,
                    height,
                )),
            ])
        tensors = [self._represent(view) for view in views]
        inputs = (
            torch.stack(tensors)
            if self.multiview
            else tensors[0]
        )
        target = (
            int(self.frame.target.iloc[index])
            if "target" in self.frame
            else -1
        )
        return inputs, torch.tensor(target), index


def make_loader(
    frame,
    transform,
    mode,
    batch_size,
    shuffle=False,
    multiview=False,
):
    '''Create a reproducible DataLoader for one expert.'''
    generator = torch.Generator()
    generator.manual_seed(SEED)
    kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": AMP,
        "drop_last": shuffle,
        "worker_init_fn": seed_worker,
        "generator": generator,
    }
    if NUM_WORKERS:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2
    dataset = ImageDataset(frame, transform, mode, multiview)
    return DataLoader(dataset, **kwargs)


preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

## 3. Modèles et moteur OOF

La tête binaire de DeepGuard est remplacée par trois sorties. Ses poids sont
transférés : la ligne `real` initialise la classe 0 et la ligne `fake`
initialise les classes 1 et 2 avec une très légère perturbation. Les deux
ConvNeXt-V2 possèdent des paramètres indépendants.


In [5]:
def unwrap(model):
    '''Return the model hidden by an optional DataParallel wrapper.'''
    if isinstance(model, nn.DataParallel):
        return model.module
    return model


class DeepModel(nn.Module):
    '''Adapt the pretrained binary DeepGuard model to three classes.'''

    def __init__(self):
        '''Load DeepGuard and transfer its binary classifier.'''
        super().__init__()
        self.net = AutoModelForImageClassification.from_pretrained(
            DEEP_NAME
        )
        classifier = self.net.classifier
        old_weight = classifier.weight.detach().clone()
        old_bias = classifier.bias.detach().clone()
        labels = {
            int(key): str(value).lower()
            for key, value in self.net.config.id2label.items()
        }
        fake_candidates = [
            key
            for key, value in labels.items()
            if "fake" in value or "manip" in value or value == "ai"
        ]
        fake_index = fake_candidates[0] if fake_candidates else 1
        real_index = 1 - fake_index
        new_head = nn.Linear(classifier.in_features, N_CLASSES)
        with torch.no_grad():
            new_head.weight[0].copy_(old_weight[real_index])
            new_head.weight[1].copy_(old_weight[fake_index])
            new_head.weight[2].copy_(old_weight[fake_index])
            new_head.weight[2].add_(
                0.001 * torch.randn_like(new_head.weight[2])
            )
            new_head.bias[0].copy_(old_bias[real_index])
            new_head.bias[1].copy_(old_bias[fake_index])
            new_head.bias[2].copy_(old_bias[fake_index])
        self.net.classifier = new_head
        self.net.config.num_labels = N_CLASSES
        self.net.config.id2label = dict(enumerate(CLASS_NAMES))
        self.net.config.label2id = {
            name: index
            for index, name in enumerate(CLASS_NAMES)
        }

    def forward(self, inputs):
        '''Return three class logits for a normalized image batch.'''
        outputs = self.net(
            pixel_values=inputs,
            return_dict=False,
        )
        return outputs[0]


class CnnModel(nn.Module):
    '''Use ConvNeXt-V2 Tiny as a three-class forensic expert.'''

    def __init__(self):
        '''Create a pretrained ConvNeXt-V2 with a new class head.'''
        super().__init__()
        self.net = timm.create_model(
            CNN_NAME,
            pretrained=True,
            num_classes=N_CLASSES,
            drop_rate=0.2,
        )

    def forward(self, inputs):
        '''Return three class logits for an RGB or NPR batch.'''
        return self.net(inputs)


def freeze_deep(model):
    '''Fine-tune the DeepGuard classifier and final visual blocks.'''
    for parameter in model.parameters():
        parameter.requires_grad = False
    for parameter in model.net.classifier.parameters():
        parameter.requires_grad = True
    vision = getattr(model.net, "vision_model", None)
    if vision is None and hasattr(model.net, "siglip"):
        vision = model.net.siglip.vision_model
    encoder = getattr(vision, "encoder", None)
    layers = getattr(encoder, "layers", [])
    for block in list(layers)[-6:]:
        for parameter in block.parameters():
            parameter.requires_grad = True
    for name in ("post_layernorm", "head"):
        module = getattr(vision, name, None)
        if module is not None:
            for parameter in module.parameters():
                parameter.requires_grad = True


def make_model(kind):
    '''Create and place one expert on the active CUDA devices.'''
    model = DeepModel() if kind == "deep" else CnnModel()
    if kind == "deep":
        freeze_deep(model)
    model = model.to(DEVICE)
    if N_GPU >= 2 and USE_TWO_GPUS:
        model = nn.DataParallel(model, device_ids=[0, 1])
    return model


def expert_settings(kind):
    '''Return transform, batch size, accumulation, and body rate.'''
    if kind == "deep":
        return deep_train_tf, BATCH_DEEP, ACCUM_DEEP, 8e-6
    return cnn_train_tf, BATCH_CNN, ACCUM_CNN, 2e-5


def predict(model, frame, kind, mode="rgb"):
    '''Return normalized three-class probabilities for every image.'''
    if kind == "deep":
        transform = deep_eval_tf
        batch_size = BATCH_DEEP
    else:
        transform = cnn_eval_tf
        batch_size = BATCH_CNN
    multiview = kind == "cnn" and mode == "npr"
    data_loader = make_loader(
        frame,
        transform,
        mode,
        batch_size,
        multiview=multiview,
    )
    output = np.zeros((len(frame), N_CLASSES), np.float32)
    model.eval()
    with torch.inference_mode():
        for inputs, _, indices in data_loader:
            if multiview:
                probabilities = []
                for view_index in range(inputs.shape[1]):
                    view = inputs[:, view_index].to(
                        DEVICE,
                        non_blocking=True,
                    )
                    with torch.autocast(
                        device_type=DEVICE,
                        dtype=torch.float16,
                        enabled=AMP,
                    ):
                        logits = model(view)
                    probabilities.append(
                        logits.float().softmax(1).cpu()
                    )
                view_probs = torch.stack(probabilities, 1)
                probs = 0.75 * view_probs.mean(1)
                probs += 0.25 * view_probs.max(1).values
                probs = probs / probs.sum(1, keepdim=True)
            else:
                inputs = inputs.to(DEVICE, non_blocking=True)
                with torch.autocast(
                    device_type=DEVICE,
                    dtype=torch.float16,
                    enabled=AMP,
                ):
                    logits = model(inputs)
                probs = logits.float().softmax(1).cpu()
            output[indices.numpy()] = probs.numpy()
    del data_loader
    return output


def macro_f1(targets, probabilities):
    '''Compute Macro F1 from a probability matrix.'''
    predictions = probabilities.argmax(1)
    return float(f1_score(targets, predictions, average="macro"))


def fit_one(kind, mode, train_part, valid_part, fold, epochs):
    '''Train one fold and restore its best Macro-F1 checkpoint.'''
    model = make_model(kind)
    transform, batch_size, accum, body_lr = expert_settings(kind)
    data_loader = make_loader(
        train_part,
        transform,
        mode,
        batch_size,
        shuffle=True,
    )
    head = []
    body = []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            destination = head if any(
                key in name.lower()
                for key in ("classifier", "head")
            ) else body
            destination.append(parameter)
    optimizer = torch.optim.AdamW([
        {"params": body, "lr": body_lr},
        {"params": head, "lr": 2e-4},
    ], weight_decay=2e-4, foreach=False)
    total_steps = max(
        1,
        math.ceil(len(data_loader) / accum) * epochs,
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        int(0.08 * total_steps),
        total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    checkpoint = OUT / (
        f"{RUN_TAG}_{kind}_{mode}_fold{fold}.pt"
    )
    best_score = -1.0
    bad_epochs = 0
    history = []
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for step, (inputs, targets, _) in enumerate(data_loader):
            inputs = inputs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE,
                dtype=torch.float16,
                enabled=AMP,
            ):
                logits = model(inputs)
                loss = F.cross_entropy(
                    logits,
                    targets,
                    label_smoothing=LABEL_SMOOTHING,
                ) / accum
            scaler.scale(loss).backward()
            final_step = step + 1 == len(data_loader)
            if (step + 1) % accum == 0 or final_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            losses.append(float(loss.detach().cpu()) * accum)
        valid_probs = predict(model, valid_part, kind, mode)
        score = macro_f1(valid_part.target.values, valid_probs)
        record = {
            "epoch": epoch + 1,
            "loss": float(np.mean(losses)),
            "macro_f1": score,
        }
        history.append(record)
        print(kind, mode, "fold", fold, record)
        if score > best_score + 1e-4:
            best_score = score
            bad_epochs = 0
            torch.save({
                "state": unwrap(model).state_dict(),
                "epoch": epoch + 1,
            }, checkpoint)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    saved = torch.load(
        checkpoint,
        map_location="cpu",
        weights_only=True,
    )
    unwrap(model).load_state_dict(saved["state"])
    # fix : force la remontée immédiate d'une éventuelle erreur CUDA
    # différée ici (exception Python normale et rattrapable), plutôt que
    # plus tard dans un thread/destructeur où elle tuerait tout le kernel.
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return model, history, int(saved["epoch"])


## 4. Prédictions OOF et holdout

Chaque ligne du train reçoit une prédiction produite sans avoir été vue par
le modèle correspondant. Les probabilités du holdout sont moyennées sur les
trois folds. Les fichiers `.npz` rendent le calcul reprenable.


In [6]:
EXPERTS = [
    ("deep", "rgb", EPOCHS_DEEP),
    ("cnn", "rgb", EPOCHS_CNN),
    ("cnn", "npr", EPOCHS_CNN),
]
n_experts = len(EXPERTS)
oof = np.zeros(
    (len(train_df), n_experts, N_CLASSES),
    np.float32,
)
hold = np.zeros(
    (len(val_df), n_experts, N_CLASSES),
    np.float32,
)
fold_epochs = {}
histories = {}

for expert_index, (kind, mode, epochs) in enumerate(EXPERTS):
    for fold, (train_index, valid_index) in enumerate(cv):
        train_part = train_df.iloc[train_index].reset_index(drop=True)
        valid_part = train_df.iloc[valid_index].reset_index(drop=True)
        cache = OUT / (
            f"{RUN_TAG}_{kind}_{mode}_fold{fold}_pred.npz"
        )
        if cache.exists() and not FORCE_RETRAIN:
            saved = np.load(cache, allow_pickle=True)
            valid_probs = saved["oof"]
            hold_probs = saved["hold"]
            best_epoch = int(saved["best_epoch"])
            history = saved["history"].tolist()
            print("RESUME", cache)
        else:
            model, history, best_epoch = fit_one(
                kind,
                mode,
                train_part,
                valid_part,
                fold,
                epochs,
            )
            valid_probs = predict(model, valid_part, kind, mode)
            hold_probs = predict(model, val_df, kind, mode)
            np.savez_compressed(
                cache,
                oof=valid_probs,
                hold=hold_probs,
                best_epoch=best_epoch,
                history=np.array(history, dtype=object),
            )
            model.cpu()
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if valid_probs.shape != (len(valid_index), N_CLASSES):
            raise ValueError("Forme OOF invalide")
        oof[valid_index, expert_index] = valid_probs
        hold[:, expert_index] += hold_probs / N_FOLDS
        key = f"{kind}_{mode}"
        histories[f"{key}_f{fold}"] = history
        fold_epochs.setdefault(key, []).append(best_epoch)
    score = macro_f1(
        train_df.target.values,
        oof[:, expert_index],
    )
    print("OOF", kind, mode, score)

if not np.isfinite(oof).all() or not np.isfinite(hold).all():
    raise ValueError("Probabilités non finies")
if not np.allclose(oof.sum(2), 1.0, atol=1e-4):
    raise ValueError("Probabilités OOF non normalisées")
np.savez_compressed(
    OUT / "level2_v2_oof.npz",
    oof=oof,
    hold=hold,
    y_train=train_df.target.values,
    y_val=val_df.target.values,
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

deep rgb fold 0 {'epoch': 1, 'loss': 0.9437996440219439, 'macro_f1': 0.7754688353790696}
deep rgb fold 0 {'epoch': 2, 'loss': 0.39118063024112154, 'macro_f1': 0.8707499789308546}
deep rgb fold 0 {'epoch': 3, 'loss': 0.28270285901042724, 'macro_f1': 0.8898772134997891}
deep rgb fold 0 {'epoch': 4, 'loss': 0.24549285700106951, 'macro_f1': 0.8914629611457832}


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

deep rgb fold 1 {'epoch': 1, 'loss': 0.9496336908247064, 'macro_f1': 0.7867248354374393}
deep rgb fold 1 {'epoch': 2, 'loss': 0.3738359826947412, 'macro_f1': 0.8745649916510727}
deep rgb fold 1 {'epoch': 3, 'loss': 0.2733222857838677, 'macro_f1': 0.8883521050068334}
deep rgb fold 1 {'epoch': 4, 'loss': 0.23654431168178808, 'macro_f1': 0.8924988638502583}


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

deep rgb fold 2 {'epoch': 1, 'loss': 0.9622495363381776, 'macro_f1': 0.7828669414583151}
deep rgb fold 2 {'epoch': 2, 'loss': 0.3934065015986562, 'macro_f1': 0.8779881467266133}
deep rgb fold 2 {'epoch': 3, 'loss': 0.26971731446683406, 'macro_f1': 0.8907434726122231}
deep rgb fold 2 {'epoch': 4, 'loss': 0.2348679811439731, 'macro_f1': 0.8932081371906225}
OOF deep rgb 0.8924222019865823


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

cnn rgb fold 0 {'epoch': 1, 'loss': 0.6890884043643125, 'macro_f1': 0.9510795228986528}
cnn rgb fold 0 {'epoch': 2, 'loss': 0.17286687990524835, 'macro_f1': 0.9780374818323012}
cnn rgb fold 0 {'epoch': 3, 'loss': 0.11871103915825852, 'macro_f1': 0.9901689775397241}
cnn rgb fold 0 {'epoch': 4, 'loss': 0.10600051336830334, 'macro_f1': 0.9905473003499531}
cnn rgb fold 1 {'epoch': 1, 'loss': 0.6499945810236346, 'macro_f1': 0.94889614516094}
cnn rgb fold 1 {'epoch': 2, 'loss': 0.16151438124289025, 'macro_f1': 0.9810511777182463}
cnn rgb fold 1 {'epoch': 3, 'loss': 0.12442965748020306, 'macro_f1': 0.9863591564639304}
cnn rgb fold 1 {'epoch': 4, 'loss': 0.09984421851499702, 'macro_f1': 0.9893979203005755}
cnn rgb fold 2 {'epoch': 1, 'loss': 0.6068835845054956, 'macro_f1': 0.9684776239189118}
cnn rgb fold 2 {'epoch': 2, 'loss': 0.15705046278229273, 'macro_f1': 0.9893990263153961}
cnn rgb fold 2 {'epoch': 3, 'loss': 0.12076065203148158, 'macro_f1': 0.9968766506181193}
cnn rgb fold 2 {'epoch': 4

## 5. Ensemble et décision robustes

Le stacker voit les neuf log-probabilités des experts. Ses prédictions de
sélection sont elles-mêmes construites en cinq folds groupés. Pour chaque
candidat, les biais additifs sont appris sur quatre folds et appliqués au
cinquième. La médiane des biais devient la règle finale.


In [7]:
def meta_features(probabilities):
    '''Flatten clipped expert log-probabilities into meta features.'''
    clipped = np.clip(probabilities, 1e-6, 1.0)
    return np.log(clipped).reshape(len(clipped), -1)


def centered_bias(values):
    '''Remove the irrelevant common offset from class biases.'''
    values = np.asarray(values, dtype=np.float64)
    return values - values.mean()


def apply_bias(probabilities, bias):
    '''Apply class biases in log space and return hard decisions.'''
    scores = np.log(np.clip(probabilities, 1e-6, 1.0))
    return (scores + centered_bias(bias)).argmax(1)


def optimize_bias(probabilities, targets):
    '''Optimize centered log-space biases for Macro F1.'''
    bias = np.zeros(N_CLASSES, dtype=np.float64)
    best = f1_score(
        targets,
        apply_bias(probabilities, bias),
        average="macro",
    )
    grids = [
        np.linspace(-0.45, 0.45, 37),
        np.linspace(-0.08, 0.08, 33),
        np.linspace(-0.02, 0.02, 33),
    ]
    for grid in grids:
        for _ in range(3):
            improved = False
            for class_index in range(N_CLASSES):
                origin = bias[class_index]
                local_value = origin
                for delta in grid:
                    trial = bias.copy()
                    trial[class_index] = origin + delta
                    trial = centered_bias(trial)
                    score = f1_score(
                        targets,
                        apply_bias(probabilities, trial),
                        average="macro",
                    )
                    if score > best + 1e-9:
                        best = score
                        local_value = trial[class_index]
                        bias = trial
                        improved = True
                bias[class_index] = local_value
            bias = centered_bias(bias)
            if not improved:
                break
    return bias, float(best)


targets = train_df.target.values
meta_cv_model = StratifiedGroupKFold(
    META_FOLDS,
    shuffle=True,
    random_state=SEED + 91,
)
meta_splits = list(
    meta_cv_model.split(oof, targets, groups)
)
meta_oof = np.zeros((len(train_df), N_CLASSES), np.float32)
for meta_train, meta_valid in meta_splits:
    meta_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.08,
            max_iter=2000,
            class_weight=None,
        ),
    )
    meta_model.fit(
        meta_features(oof[meta_train]),
        targets[meta_train],
    )
    meta_oof[meta_valid] = meta_model.predict_proba(
        meta_features(oof[meta_valid])
    )

stack = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.08,
        max_iter=2000,
        class_weight=None,
    ),
)
stack.fit(meta_features(oof), targets)
stack_hold = stack.predict_proba(meta_features(hold))

candidates = {
    "stack": meta_oof,
    "mean": oof.mean(1),
}
candidate_hold = {
    "stack": stack_hold,
    "mean": hold.mean(1),
}
for expert_index, (kind, mode, _) in enumerate(EXPERTS):
    name = f"{kind}_{mode}"
    candidates[name] = oof[:, expert_index]
    candidate_hold[name] = hold[:, expert_index]

candidate_scores = {}
candidate_biases = {}
candidate_fold_scores = {}
for name, probabilities in candidates.items():
    cross_predictions = np.zeros(len(train_df), dtype=int)
    biases = []
    fold_scores = []
    for meta_train, meta_valid in meta_splits:
        bias, _ = optimize_bias(
            probabilities[meta_train],
            targets[meta_train],
        )
        biases.append(bias)
        predictions = apply_bias(
            probabilities[meta_valid],
            bias,
        )
        cross_predictions[meta_valid] = predictions
        fold_scores.append(float(f1_score(
            targets[meta_valid],
            predictions,
            average="macro",
        )))
    candidate_scores[name] = float(f1_score(
        targets,
        cross_predictions,
        average="macro",
    ))
    candidate_biases[name] = centered_bias(
        np.median(np.stack(biases), axis=0)
    )
    candidate_fold_scores[name] = fold_scores

SELECTED = max(candidate_scores, key=candidate_scores.get)
FINAL_BIAS = candidate_biases[SELECTED]
oof_macro_f1 = candidate_scores[SELECTED]
hold_probs = candidate_hold[SELECTED]
hold_predictions = apply_bias(hold_probs, FINAL_BIAS)
hold_targets = val_df.target.values
hold_macro_f1 = float(f1_score(
    hold_targets,
    hold_predictions,
    average="macro",
))
hold_balanced = float(balanced_accuracy_score(
    hold_targets,
    hold_predictions,
))
hold_per_class = f1_score(
    hold_targets,
    hold_predictions,
    average=None,
    labels=np.arange(N_CLASSES),
).tolist()
hold_matrix = confusion_matrix(
    hold_targets,
    hold_predictions,
    labels=np.arange(N_CLASSES),
).tolist()

print("OOF candidates", candidate_scores)
print("SELECTED", SELECTED, "bias", FINAL_BIAS.round(5))
print("OOF Macro F1", oof_macro_f1)
print("UNTOUCHED HOLDOUT Macro F1", hold_macro_f1)
print(classification_report(
    hold_targets,
    hold_predictions,
    target_names=CLASS_NAMES,
    digits=5,
))
if hold_macro_f1 < oof_macro_f1 - 0.015:
    print("WARNING: écart OOF/holdout supérieur à 1,5 point")


OOF candidates {'stack': 0.9970740749104224, 'mean': 0.9969466062905455, 'deep_rgb': 0.8934422796367762, 'cnn_rgb': 0.9929983063934079, 'cnn_npr': 0.9950370257145839}
SELECTED stack bias [-0.19167 -0.19167  0.38333]
OOF Macro F1 0.9970740749104224
UNTOUCHED HOLDOUT Macro F1 0.9959115718380649
              precision    recall  f1-score   support

        real    1.00000   0.99387   0.99692       326
   generated    0.99388   0.99693   0.99541       326
  local_edit    0.99388   0.99693   0.99541       326

    accuracy                        0.99591       978
   macro avg    0.99592   0.99591   0.99591       978
weighted avg    0.99592   0.99591   0.99591       978



## 6. Réentraînement final et inférence test

Le nombre d'epochs de chaque expert est la médiane des checkpoints OOF.
Les modèles finaux apprennent sur `train + val`. Le stacker, le candidat et
les biais restent ceux qui ont été figés avant l'audit du holdout.


In [8]:
all_df = pd.concat([train_df, val_df], ignore_index=True)
final_test = np.zeros(
    (len(test_df), n_experts, N_CLASSES),
    np.float32,
)

for expert_index, (kind, mode, _) in enumerate(EXPERTS):
    key = f"{kind}_{mode}"
    epochs = max(1, int(round(np.median(fold_epochs[key]))))
    checkpoint = OUT / f"{RUN_TAG}_final_{kind}_{mode}.pt"
    model = make_model(kind)
    if checkpoint.exists() and not FORCE_RETRAIN:
        unwrap(model).load_state_dict(torch.load(
            checkpoint,
            map_location="cpu",
            weights_only=True,
        ))
        print("RESUME", checkpoint)
        final_test[:, expert_index] = predict(
            model,
            test_df,
            kind,
            mode,
        )
        model.cpu()
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    transform, batch_size, accum, body_lr = expert_settings(kind)
    data_loader = make_loader(
        all_df,
        transform,
        mode,
        batch_size,
        shuffle=True,
    )
    head = []
    body = []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            destination = head if any(
                token in name.lower()
                for token in ("classifier", "head")
            ) else body
            destination.append(parameter)
    optimizer = torch.optim.AdamW([
        {"params": body, "lr": body_lr},
        {"params": head, "lr": 2e-4},
    ], weight_decay=2e-4, foreach=False)
    total_steps = max(
        1,
        math.ceil(len(data_loader) / accum) * epochs,
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        int(0.08 * total_steps),
        total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for step, (inputs, labels, _) in enumerate(data_loader):
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.autocast(
                device_type=DEVICE,
                dtype=torch.float16,
                enabled=AMP,
            ):
                logits = model(inputs)
                loss = F.cross_entropy(
                    logits,
                    labels,
                    label_smoothing=LABEL_SMOOTHING,
                ) / accum
            scaler.scale(loss).backward()
            final_step = step + 1 == len(data_loader)
            if (step + 1) % accum == 0 or final_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            losses.append(float(loss.detach().cpu()) * accum)
        print(
            "FINAL",
            kind,
            mode,
            "epoch",
            epoch + 1,
            "loss",
            float(np.mean(losses)),
        )
    torch.save(unwrap(model).state_dict(), checkpoint)
    final_test[:, expert_index] = predict(
        model,
        test_df,
        kind,
        mode,
    )
    model.cpu()
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

candidate_test = {
    "stack": stack.predict_proba(meta_features(final_test)),
    "mean": final_test.mean(1),
}
for expert_index, (kind, mode, _) in enumerate(EXPERTS):
    candidate_test[f"{kind}_{mode}"] = final_test[:, expert_index]
test_probs = candidate_test[SELECTED]
test_predictions = apply_bias(test_probs, FINAL_BIAS)


Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

FINAL deep rgb epoch 1 loss 0.7949376357516841
FINAL deep rgb epoch 2 loss 0.3067178132680609
FINAL deep rgb epoch 3 loss 0.22644629430673693
FINAL deep rgb epoch 4 loss 0.19802615840149962
FINAL cnn rgb epoch 1 loss 0.5037619633114209
FINAL cnn rgb epoch 2 loss 0.13313245357004116
FINAL cnn rgb epoch 3 loss 0.10596273958419813
FINAL cnn rgb epoch 4 loss 0.09766922138742698
FINAL cnn npr epoch 1 loss 0.4848866396274863
FINAL cnn npr epoch 2 loss 0.16573663415265796
FINAL cnn npr epoch 3 loss 0.12023277711325689
FINAL cnn npr epoch 4 loss 0.11060755926882848


## 7. Soumission et audit reproductible

Les identifiants sont joints au sample officiel plutôt que supposés dans le
même ordre. L'audit conserve les scores, choix, epochs, versions et durée.


In [9]:
sample = pd.read_csv(ROOT / "sample_submission.csv")
predictions = pd.DataFrame({
    "image_id": test_df.image_id,
    "classe": test_predictions,
})
submission = sample[["image_id"]].merge(
    predictions,
    on="image_id",
    how="left",
    validate="one_to_one",
)
if len(submission) != len(sample):
    raise ValueError("Nombre de lignes incorrect")
if submission.classe.isna().any():
    raise ValueError("Prédictions manquantes")
if not submission.image_id.is_unique:
    raise ValueError("Identifiants dupliqués")
submission.classe = submission.classe.astype(int)
if not set(submission.classe.unique()) <= {0, 1, 2}:
    raise ValueError("Classes invalides")
submission.to_csv(OUT / "submission.csv", index=False)

elapsed_hours = (time.monotonic() - START_TIME) / 3600
audit = {
    "version": "level2_v2_clean_oof_hybrid",
    "seed": SEED,
    "root": str(ROOT),
    "n_folds": N_FOLDS,
    "models": [
        DEEP_NAME,
        CNN_NAME + " RGB",
        CNN_NAME + " NPR",
    ],
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "validation_policy": (
        "grouped train OOF only for selection; official validation "
        "opened once; final refit on train plus validation"
    ),
    "selected_candidate": SELECTED,
    "candidate_oof_macro_f1": candidate_scores,
    "candidate_fold_macro_f1": candidate_fold_scores,
    "decision_bias": FINAL_BIAS.tolist(),
    "oof_macro_f1": oof_macro_f1,
    "untouched_validation_macro_f1": hold_macro_f1,
    "untouched_validation_f1_per_class": hold_per_class,
    "untouched_validation_balanced_accuracy": hold_balanced,
    "untouched_validation_confusion_matrix": hold_matrix,
    "best_epochs": fold_epochs,
    "histories": histories,
    "predicted_class_counts": {
        str(key): int(value)
        for key, value in pd.Series(test_predictions)
        .value_counts()
        .sort_index()
        .items()
    },
    "elapsed_hours": elapsed_hours,
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "timm": timm.__version__,
        "transformers": transformers.__version__,
        "scikit_learn": sklearn.__version__,
    },
}
audit_path = OUT / "audit_level2_v2.json"
audit_path.write_text(json.dumps(audit, indent=2))
np.savez_compressed(
    OUT / "ensemble_scores_level2_v2.npz",
    oof=oof,
    hold=hold,
    test=final_test,
    meta_oof=meta_oof,
    selected_hold=hold_probs,
    selected_test=test_probs,
    y_train=targets,
    y_val=hold_targets,
)
display(submission.head())
display(pd.DataFrame(hold_matrix, index=CLASS_NAMES, columns=CLASS_NAMES))
print(json.dumps({
    key: value
    for key, value in audit.items()
    if key != "histories"
}, indent=2))
print("READY", OUT / "submission.csv")


,image_id,classe
0,e37001f6-d230-4588-aedb-bd896084d671.png,2
1,56689e24-fe39-4df5-be2e-e0669e30d1d6.png,2
2,0c9f82ad-a7f5-4ee0-9f20-90333608428a.png,0
3,d5471932-3e03-45ea-a3bd-916be674b2ca.png,1
4,b8d8e96c-ca54-41d7-a2ab-a713e5a79d2f.png,0


,real,generated,local_edit
real,324,1,1
generated,0,325,1
local_edit,0,1,325


{
  "version": "level2_v2_clean_oof_hybrid",
  "seed": 20260726,
  "root": "/kaggle/input/competitions/gensivana-real-or-fake-level-2-classification",
  "n_folds": 3,
  "models": [
    "king1oo1/deepfake-model",
    "convnextv2_tiny.fcmae_ft_in22k_in1k_384 RGB",
    "convnextv2_tiny.fcmae_ft_in22k_in1k_384 NPR"
  ],
  "n_train": 7857,
  "n_val": 978,
  "n_test": 993,
  "validation_policy": "grouped train OOF only for selection; official validation opened once; final refit on train plus validation",
  "selected_candidate": "stack",
  "candidate_oof_macro_f1": {
    "stack": 0.9970740749104224,
    "mean": 0.9969466062905455,
    "deep_rgb": 0.8934422796367762,
    "cnn_rgb": 0.9929983063934079,
    "cnn_npr": 0.9950370257145839
  },
  "candidate_fold_macro_f1": {
    "stack": [
      0.9987456871935017,
      0.9949497945699268,
      0.9974345934097747,
      0.9960999500255151,
      0.9980876936153417
    ],
    "mean": [
      0.9987456871935017,
      0.9943149013303362,
      0.99

## Critère de lecture du résultat

Le score public ne doit pas servir à réoptimiser les biais. La référence de
décision est le Macro F1 du holdout officiel, accompagné des trois F1 par
classe et de l'écart avec les OOF. Un score public isolé peut varier fortement
puisque le leaderboard public ne couvre qu'une petite fraction du test.
